In [ ]:
# I'm importing the BigQuery library, which I'll need to create my table.
from google.cloud import bigquery
import time
from google.cloud import pubsub_v1

# I'll define my specific project ID and the region for this lab.
my_project_id = "qwiklabs-gcp-01-ed33dcbea5fb"

# Now, I'll initialize the BigQuery client so I can run commands.
client = bigquery.Client(project=my_project_id)

print(f"My environment is set up and ready for Challenge 4, for project: {my_project_id}")


My environment is set up and ready for Challenge 4, for project: qwiklabs-gcp-01-ed33dcbea5fb


In [ ]:
# I'll define the names for my new dataset and table to keep things organized.
dataset_name = "live_flight_data"
table_name = "realtime_transponder_feed"

# I'm creating the dataset in the 'US' location. ---
dataset_id = f"{my_project_id}.{dataset_name}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"
client.create_dataset(dataset, exists_ok=True) # 'exists_ok=True' prevents errors if I run this again.
print(f"I have successfully created the dataset: {dataset_name}")

# I'm defining the exact schema for my table. ---
# This structure must perfectly match the incoming streaming data.
table_schema = [
    bigquery.SchemaField("MT", "STRING"), bigquery.SchemaField("TT", "INT64"),
    bigquery.SchemaField("SID", "STRING"), bigquery.SchemaField("AID", "STRING"),
    bigquery.SchemaField("Hex", "STRING"), bigquery.SchemaField("FID", "STRING"),
    bigquery.SchemaField("DMG", "DATE"), bigquery.SchemaField("TMG", "TIME"),
    bigquery.SchemaField("DML", "DATE"), bigquery.SchemaField("TML", "TIME"),
    bigquery.SchemaField("CS", "STRING"), bigquery.SchemaField("Alt", "INT64"),
    bigquery.SchemaField("GS", "FLOAT64"), bigquery.SchemaField("Trk", "FLOAT64"),
    bigquery.SchemaField("Lat", "FLOAT64"), bigquery.SchemaField("Lng", "FLOAT64"),
    bigquery.SchemaField("VR", "INT64"), bigquery.SchemaField("Sq", "STRING"),
    bigquery.SchemaField("Alrt", "INT64"), bigquery.SchemaField("Emer", "INT64"),
    bigquery.SchemaField("SPI", "INT64"), bigquery.SchemaField("Gnd", "INT64"),
]

# I'm creating the table with the schema I just defined. ---
table_id = f"{dataset_id}.{table_name}"
table = bigquery.Table(table_id, schema=table_schema)
client.create_table(table, exists_ok=True)

print(f"I have successfully created the table: {table_name}")


I have successfully created the dataset: live_flight_data
I have successfully created the table: realtime_transponder_feed


In [ ]:
#  I'll define all my configuration details. ---
source_project_id = "paul-leroy"                 # The project where the data is being published.
topic_name = "flight-transponder"
my_subscription_id = "my-flight-data-subscriber" # My unique name for the subscription.
my_table_id = f"{my_project_id}.live_flight_data.realtime_transponder_feed"

#  I'll set up my Pub/Sub and BigQuery clients. ---
subscriber = pubsub_v1.SubscriberClient()
bq_client = bigquery.Client(project=my_project_id)

# I'll create my subscription to the source topic.
topic_path = f"projects/{source_project_id}/topics/{topic_name}"
subscription_path = subscriber.subscription_path(my_project_id, my_subscription_id)

try:
    subscriber.create_subscription(name=subscription_path, topic=topic_path)
    print(f"I have successfully created my subscription: {my_subscription_id}")
except Exception as e:
    print(f"My subscription status: {e}") # This is okay, it just means the subscription already exists.

#  I'll define my 'callback' function to process each message. ---
def process_message_callback(message):
    try:
        # I'll decode the message and split the comma-separated text into a list.
        data_points = message.data.decode("utf-8").split(",")
        column_names = ["MT", "TT", "SID", "AID", "Hex", "FID", "DMG", "TMG", "DML", "TML",
                        "CS", "Alt", "GS", "Trk", "Lat", "Lng", "VR", "Sq", "Alrt", "Emer", "SPI", "Gnd"]

        # I'll create a dictionary to hold my row data.
        row_to_insert = {}
        for i in range(len(column_names)):
            value = data_points[i] if i < len(data_points) and data_points[i] != '' else None
            # BigQuery needs dates as YYYY-MM-DD, so I'll replace the slashes.
            if value and column_names[i] in ["DMG", "DML"]:
                value = value.replace("/", "-")
            row_to_insert[column_names[i]] = value

        # I'll insert the prepared row into my BigQuery table.
        errors = bq_client.insert_rows_json(my_table_id, [row_to_insert])
        if not errors:
            # If successful, I'll acknowledge the message so it's not sent again.
            message.ack()
    except Exception:
        pass
#  I'll start listening for messages. ---
streaming_pull_future = subscriber.subscribe(subscription_path, callback=process_message_callback)
print(f"I am now listening for live flight data from: {topic_path}...")
print("I will let this run for 3 minutes to collect a good sample of data. Please wait...")

# I'll keep this script running for 180 seconds (3 minutes).
try:
    streaming_pull_future.result(timeout=180)
except Exception:
    streaming_pull_future.cancel()
    print("\nMy data collection is complete.")


I have successfully created my subscription: my-flight-data-subscriber
I am now listening for live flight data from: projects/paul-leroy/topics/flight-transponder...
I will let this run for 3 minutes to collect a good sample of data. Please wait...

My data collection is complete.


In [ ]:
%%bigquery
-- First, I'll count the total number of messages I received.
SELECT
  COUNT(*) as total_records_captured
FROM
  `qwiklabs-gcp-01-ed33dcbea5fb.live_flight_data.realtime_transponder_feed`;


Query is running:   0%|          |

Downloading:   0%|          |

,total_records_captured
0,49052
